# Revisión capa SILVER (datos limpios + modelo estrella)

**Regla de oro de integridad**: `bronce = limpios (stage) + rechazados (reject)`
y `facts = limpios` — ninguna fila se pierde, solo se clasifica.
Los rechazados no se eliminan: quedan documentados con su motivo.

In [1]:
from pathlib import Path

import polars as pl
import pyarrow.parquet as pq

DATA = Path("../data") if Path("../data").exists() else Path("data")
CATS = ["green", "yellow", "fhv", "fhvhv"]
YEARS = [2023, 2024, 2025]
MESES = [(y, m) for y in YEARS for m in range(1, 13)]
pl.Config.set_tbl_rows(80)
pl.Config.set_tbl_cols(25)


def parquet_rows(path: Path) -> int | None:
    """Filas totales de un archivo/directorio parquet leyendo solo metadatos.

    Ojo: Spark escribe "archivos" mensuales como DIRECTORIOS llamados *.parquet
    con part-files dentro — por eso se filtra con is_file(). Se ignora la
    basura _temporary y los archivos de 0 bytes que deja un job interrumpido.
    """
    if not path.exists():
        return None
    files = (
        [path]
        if path.is_file()
        else [
            f for f in path.rglob("*.parquet")
            if f.is_file() and f.stat().st_size > 0 and "_temporary" not in str(f)
        ]
    )
    if not files:
        return None
    return sum(pq.ParquetFile(f).metadata.num_rows for f in files)


def dir_mb(path: Path) -> float:
    """Peso en MB de un archivo o directorio."""
    if not path.exists():
        return 0.0
    files = [path] if path.is_file() else [f for f in path.rglob("*") if f.is_file()]
    return round(sum(f.stat().st_size for f in files) / 1024**2, 2)

## Verificación de integridad archivo por archivo (144 meses)

In [2]:
filas = []
for cat in CATS:
    for y, m in MESES:
        b = parquet_rows(DATA / "bronze" / cat / f"{y}-{m:02d}.parquet")
        s = parquet_rows(DATA / "silver" / "stage" / cat / f"{y}-{m:02d}.parquet") or 0
        r = parquet_rows(DATA / "silver" / "reject" / cat / f"{y}-{m:02d}.parquet") or 0
        f = parquet_rows(DATA / "silver" / "star" / "facts" / f"fact_{cat}_trip" / f"{y}-{m:02d}.parquet") or 0
        filas.append({
            "categoria": cat, "mes": f"{y}-{m:02d}", "bronce": b, "limpios": s,
            "rechazados": r, "facts": f,
            "cuadra": (b == s + r) and (f == s),
        })
integ = pl.DataFrame(filas)
tot = integ.select(pl.col("bronce").sum(), pl.col("limpios").sum(), pl.col("rechazados").sum(), pl.col("facts").sum())
print(f"TOTALES -> bronce: {tot[0,0]:,} | limpios: {tot[0,1]:,} | rechazados: {tot[0,2]:,} | facts: {tot[0,3]:,}")
print(f"bronce == limpios + rechazados: {tot[0,0] == tot[0,1] + tot[0,2]}")
print(f"facts  == limpios:              {tot[0,3] == tot[0,1]}")
descuadres = integ.filter(~pl.col("cuadra"))
print(f"archivos con descuadre: {descuadres.height}")
descuadres if descuadres.height else integ.head(12)

TOTALES -> bronce: 904,327,862 | limpios: 838,153,259 | rechazados: 66,174,603 | facts: 838,153,259
bronce == limpios + rechazados: True
facts  == limpios:              True
archivos con descuadre: 0


categoria,mes,bronce,limpios,rechazados,facts,cuadra
str,str,i64,i64,i64,i64,bool
"""green""","""2023-01""",68211,63709,4502,63709,true
"""green""","""2023-02""",64809,59775,5034,59775,true
"""green""","""2023-03""",72044,67301,4743,67301,true
"""green""","""2023-04""",65392,60730,4662,60730,true
"""green""","""2023-05""",69174,64265,4909,64265,true
"""green""","""2023-06""",65550,60181,5369,60181,true
"""green""","""2023-07""",61343,56443,4900,56443,true
"""green""","""2023-08""",60649,56567,4082,56567,true
"""green""","""2023-09""",65471,60613,4858,60613,true


## Motivos de rechazo (la capa de calidad trabajando)

Cada fila rechazada conserva su `_reject_reason`: son datos inválidos
documentados (duplicados, fechas fuera de periodo, zonas inexistentes...).

In [3]:
# Se lee SOLO _reject_reason mes a mes: el esquema completo varia por anio
# (p.ej. cbd_congestion_fee existe solo desde 2025) y un scan global fallaria.
rechazos = []
for cat in CATS:
    for y, m in MESES:
        d = DATA / "silver" / "reject" / cat / f"{y}-{m:02d}.parquet"
        if not d.exists():
            continue
        df = pl.read_parquet(str(d / "**" / "*.parquet"), columns=["_reject_reason"])
        if df.height:
            rechazos.append(
                df.group_by("_reject_reason").agg(pl.len().alias("filas"))
                  .with_columns(pl.lit(cat).alias("categoria"))
            )
rej = (pl.concat(rechazos)
         .group_by("_reject_reason", "categoria")
         .agg(pl.col("filas").sum().alias("filas")))
total_bronce = 904_327_862  # verificado 2026-07-03: 144 archivos, 3 años
print(f"total rechazados: {rej['filas'].sum():,} ({rej['filas'].sum() / total_bronce * 100:.3f}% del bronce)")
rej.pivot(values="filas", index="_reject_reason", on="categoria").sort("_reject_reason")

total rechazados: 66,174,603 (7.318% del bronce)


_reject_reason,yellow,fhv,green,fhvhv
str,u32,u32,u32,u32
"""datetime_inverted""",2869,163,5,30338
"""incomplete_DOlocationID""",null,221154,null,null
"""incomplete_PUlocationID""",null,46919807,null,null
"""incomplete_passenger_count""",17012482,null,129821,null
"""timeliness_off_period""",1364,null,553,null
"""uniqueness_duplicate""",1791320,50421,6129,8177


## Modelo estrella: dimensiones

In [4]:
dims_dir = DATA / "silver" / "star" / "dims"
pl.DataFrame([
    {"dimension": p.stem, "filas": parquet_rows(p), "peso_mb": dir_mb(p)}
    for p in sorted(dims_dir.glob("*.parquet"))
])

dimension,filas,peso_mb
str,i64,f64
"""dim_date""",1096,0.04
"""dim_payment_type""",8,0.01
"""dim_ratecode""",7,0.01
"""dim_service""",4,0.01
"""dim_vendor""",5,0.0
"""dim_zone""",265,0.0


## Modelo estrella: tablas de hechos

In [5]:
pl.DataFrame([
    {"fact": f"fact_{cat}_trip", "filas": parquet_rows(DATA / "silver" / "star" / "facts" / f"fact_{cat}_trip"),
     "peso_mb": dir_mb(DATA / "silver" / "star" / "facts" / f"fact_{cat}_trip")}
    for cat in CATS
])

fact,filas,peso_mb
str,i64,f64
"""fact_green_trip""",1902145,57.44
"""fact_yellow_trip""",111743703,2886.67
"""fact_fhv_trip""",11344964,261.41
"""fact_fhvhv_trip""",715511637,26485.08
